# 🎙️ Fine-tune Wav2Vec2 Nhận Dạng Tiếng Việt
**Dataset:** `linhtran92/viet_bud500` (~500 giờ, đa dạng accent)  
**Base model:** `nguyenvulebinh/wav2vec2-base-vietnamese-250h`  
**GPU:** Kaggle T4 x2 (16GB VRAM)  
**Metric:** WER (Word Error Rate) — càng thấp càng tốt

## Cell 1 — Cài đặt thư viện

In [ ]:
!pip install -q transformers datasets evaluate jiwer accelerate soundfile librosa

## Cell 2 — Import & Kiểm tra GPU

In [ ]:
import os
import torch
import numpy as np
from pathlib import Path
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union

import evaluate
from datasets import load_dataset, Audio
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

# Tối ưu bộ nhớ CUDA
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

# Kiểm tra GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu_name}")
    print(f"✅ VRAM: {vram:.1f} GB")
    print(f"✅ Số GPU: {torch.cuda.device_count()}")
else:
    print("❌ Không tìm thấy GPU — kiểm tra lại Kaggle Settings > Accelerator")

print(f"✅ PyTorch version: {torch.__version__}")

## Cell 3 — Load & Lọc Dataset từ HuggingFace

In [ ]:
print("⏳ Đang load dataset linhtran92/viet_bud500 từ HuggingFace...")
print("   (Lần đầu sẽ mất vài phút để tải về)")

ds = load_dataset("linhtran92/viet_bud500")
print("✅ Load dataset thành công!")
print(ds)

# ── Lọc audio quá ngắn (< 1s) khỏi split train ──
# Kết quả check_dataset.py cho thấy có 744 samples < 1s cần loại bỏ
print("\n⏳ Đang lọc samples audio quá ngắn (< 1s) khỏi split train...")

def is_audio_valid(sample):
    """Trả về True nếu audio đủ dài (>= 1 giây)"""
    audio_array = sample["audio"]["array"]
    sample_rate = sample["audio"]["sampling_rate"]
    duration = len(audio_array) / sample_rate
    return duration >= 1.0

train_before = len(ds["train"])
ds["train"] = ds["train"].filter(is_audio_valid, num_proc=2)
train_after = len(ds["train"])

print(f"✅ Train trước khi lọc : {train_before:,} samples")
print(f"✅ Train sau khi lọc  : {train_after:,} samples")
print(f"🗑️  Đã loại bỏ         : {train_before - train_after:,} samples")

# ── Resample tất cả audio về 16000 Hz ──
print("\n⏳ Đang resample audio về 16000 Hz...")
ds = ds.cast_column("audio", Audio(sampling_rate=16000))
print("✅ Resample hoàn tất!")
print(f"   Train: {len(ds['train']):,} | Validation: {len(ds['validation']):,} | Test: {len(ds['test']):,}")

## Cell 4 — Load Processor & Chuẩn bị Data

In [ ]:
MODEL_NAME = "nguyenvulebinh/wav2vec2-base-vietnamese-250h"

print(f"⏳ Đang load processor từ {MODEL_NAME}...")
processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
print("✅ Processor loaded!")
print(f"   Vocab size: {len(processor.tokenizer)}")

def prepare_dataset(batch):
    """Chuẩn bị một batch: extract audio + tokenize transcript"""
    # Lấy audio array (đã resample 16kHz)
    audio = [a["array"] for a in batch["audio"]]
    
    # Trích xuất input_values từ audio
    batch["input_values"] = processor(
        audio,
        sampling_rate=16000,
        return_tensors=None,  # trả về list để batch
    ).input_values
    
    # Tokenize transcript (đã 100% lowercase, không cần normalize thêm)
    with processor.as_target_processor():
        batch["labels"] = processor(
            batch["transcription"]
        ).input_ids
    
    return batch

print("\n⏳ Đang chuẩn bị dataset (map feature extraction)...")
print("   Quá trình này có thể mất 20-40 phút với dataset lớn")

ds = ds.map(
    prepare_dataset,
    remove_columns=ds["train"].column_names,  # xóa cột gốc
    batched=True,
    batch_size=32,
    num_proc=2,
)

print("✅ Chuẩn bị dataset hoàn tất!")
print(f"   Các cột sau map: {ds['train'].column_names}")

## Cell 5 — Load Model & Data Collator

In [ ]:
@dataclass
class DataCollatorCTCWithPadding:
    """
    Data collator cho CTC: padding input_values và labels riêng biệt.
    Labels dùng -100 để đánh dấu padding (bị ignore khi tính loss).
    """
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True
    pad_to_multiple_of: Optional[int] = 8  # tối ưu tensor core T4
    pad_to_multiple_of_labels: Optional[int] = 8

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        # Padding audio
        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        # Padding labels
        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(
                label_features,
                padding=self.padding,
                pad_to_multiple_of=self.pad_to_multiple_of_labels,
                return_tensors="pt",
            )

        # Thay padding token bằng -100 để loss function bỏ qua
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels
        return batch


# Khởi tạo data collator
data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

# Load model
print(f"⏳ Đang load model từ {MODEL_NAME}...")
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    attention_dropout=0.1,
    hidden_dropout=0.1,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,   # giảm xuống để training ổn định hơn
    layerdrop=0.0,
)

# Freeze feature extractor để chỉ fine-tune phần encoder trở lên
model.freeze_feature_extractor()
print("✅ Model loaded & feature extractor frozen!")

# Đếm params trainable
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"   Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

## Cell 6 — Định nghĩa Metrics & Training Arguments

In [ ]:
# Load WER metric
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    """Tính WER trên tập validation sau mỗi eval_steps"""
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    # Thay -100 về pad_token_id trước khi decode
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    # Decode predictions và references
    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    print(f"   📊 WER: {wer:.4f} ({wer*100:.2f}%)")
    return {"wer": wer}


# Output directory trên Kaggle
OUTPUT_DIR = "/kaggle/working/wav2vec2-viet-bud500"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    # ── Batch & Gradient ──
    per_device_train_batch_size=16,      # T4 16GB chịu được batch 16
    gradient_accumulation_steps=2,       # effective batch = 32
    # ── Learning rate ──
    learning_rate=1e-4,
    warmup_steps=500,
    # ── Steps ──
    max_steps=20000,
    # ── Evaluation & Saving ──
    eval_strategy="steps",
    eval_steps=1000,
    save_steps=1000,
    save_total_limit=2,                  # chỉ giữ 2 checkpoint gần nhất
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,             # WER càng thấp càng tốt
    # ── Performance ──
    fp16=True,                           # bắt buộc để tiết kiệm VRAM
    group_by_length=True,                # gom batch cùng độ dài → giảm padding
    dataloader_num_workers=2,
    # ── Logging ──
    logging_steps=100,
    logging_dir=f"{OUTPUT_DIR}/logs",
    report_to="none",                    # tắt wandb để tránh lỗi
)

print("✅ Training arguments đã được cấu hình!")
print(f"   Output dir    : {OUTPUT_DIR}")
print(f"   Max steps     : {training_args.max_steps:,}")
print(f"   Batch size    : {training_args.per_device_train_batch_size} x {training_args.gradient_accumulation_steps} = {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps} (effective)")
print(f"   Learning rate : {training_args.learning_rate}")
print(f"   fp16          : {training_args.fp16}")

## Cell 7 — Khởi tạo Trainer & Bắt đầu Training

In [ ]:
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    tokenizer=processor.feature_extractor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

# Kiểm tra có checkpoint cũ để resume không
checkpoint = None
if Path(OUTPUT_DIR).exists():
    checkpoints = sorted(Path(OUTPUT_DIR).glob("checkpoint-*"))
    if checkpoints:
        checkpoint = str(checkpoints[-1])
        print(f"🔄 Tìm thấy checkpoint: {checkpoint}")
        print("   Sẽ resume training từ checkpoint này")
    else:
        print("🆕 Không có checkpoint cũ — bắt đầu training từ đầu")
else:
    print("🆕 Bắt đầu training từ đầu")

# Giải phóng VRAM trước khi training
torch.cuda.empty_cache()
print(f"\n🚀 Bắt đầu training...")
print(f"   Ước tính thời gian: 10-15 giờ trên T4 x2")
print(f"   WER sẽ được in ra mỗi {training_args.eval_steps} steps\n")

try:
    trainer.train(resume_from_checkpoint=checkpoint)
    print("\n✅ Training hoàn tất!")
except RuntimeError as e:
    if "CUDA out of memory" in str(e):
        print("\n❌ CUDA OOM! Hãy thử:")
        print("   1. Giảm per_device_train_batch_size xuống 8")
        print("   2. Tăng gradient_accumulation_steps lên 4")
        print("   3. Restart kernel và chạy lại")
    else:
        raise e

## Cell 8 — Lưu Model & Đóng gói Download

In [ ]:
import shutil

# Đánh giá lần cuối trên tập validation
print("⏳ Đang đánh giá model trên tập validation...")
eval_results = trainer.evaluate()
final_wer = eval_results.get('eval_wer', 'N/A')
print(f"\n🏆 WER cuối cùng trên validation: {final_wer}")
if isinstance(final_wer, float):
    print(f"   ({final_wer*100:.2f}%)")

# Lưu model + processor
print(f"\n⏳ Đang lưu model vào {OUTPUT_DIR}...")
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print("✅ Model & processor đã được lưu!")

# Đóng gói thành file zip để download về máy
ZIP_PATH = "/kaggle/working/model_final"
print(f"\n⏳ Đang đóng gói model thành file zip...")
shutil.make_archive(ZIP_PATH, "zip", OUTPUT_DIR)
zip_size = Path(ZIP_PATH + ".zip").stat().st_size / 1e6
print(f"✅ Đóng gói hoàn tất!")
print(f"   📦 File: {ZIP_PATH}.zip")
print(f"   📦 Size: {zip_size:.1f} MB")
print(f"\n👉 Vào tab Output bên phải Kaggle để download file model_final.zip")